In [35]:
import numpy as np

In [38]:
# Material Properties
ba_den = 3.51 # g/cm^3
mg_den = 1.738 # g/cm^3
ba_m = 137.33 # g/mol
mg_m = 24.305 # g/mol

# Source Stats: 20% barium and 80% magnesium
w_ba, w_mg = 0.2, 0.8
source_density = 1.0/(w_ba/ba_den + w_mg/mg_den) # g/cm^3
ba_partial_density = source_density * w_ba # g/cm^3
mg_partial_density = source_density * w_mg # g/cm^3

# Thermal Properties (specific heat capacity (cp) and latent heat of vaporization (Lv)
cp_ba, cp_mg = 204.0, 1023.0 # J/kg*K
Lv_ba_molar, Lv_mg_molar = 150e3, 127.4e3 # J/mol
Lv_ba = Lv_ba_molar / (ba_m * 1e-3) # J/kg
Lv_mg = Lv_mg_molar / (mg_m * 1e-3) # J/kg
cp_mix = w_ba * cp_ba + w_mg * cp_mg # J/kg*K
Lv_mix = w_ba * Lv_ba + w_mg * Lv_mg # J/kg

T0 = 293 # K
Tv = 2000 # K - (working vaporization temperature estimate)
Tm = 923 # k - (melting point placeholder (pure Mg estimate))

# Thermal Diffusivity (Mg-dominated)
k_mg, rho_mg, cp_mg = 156, 1738, 1023 # W/m*K, kg/m^3, J/kg*K
D_thermal = k_mg / (rho_mg * cp_mg) # m^2/s

# Laser parameters (MNL 100)
wavelength = 337.1 #nm
pulse_duration = 3e-9 # Pulse duration (s)
rate = 20 # Pulse rate (Hz)
E_pulse_max = 130e-6 #J

# Physical Constants
n_a = 6.022e23 # Avogadro's number
k_b = 1.38065e-16 # Boltzmann's constant (g*cm^2/s^2*K)
T = 293 # K

def threshold_fluence():
    """
    Citable lower-bound threshold (Marks et al. 2022, Precision Engineering):
    F_th_melt = rho * c * (T_m - T_0) * l_th / (1 - R) - Lower Bound
    rho: kg/m^3, c: J/kg/K, T_m, T_0: K, D: thermal diffusivity m^2/s,
    tau: pulse duration (s), R: reflectivity (0-1)
    """

    A = 0.13

    L_th_m = np.sqrt(4* D_thermal * pulse_duration) # thermal diffusion length, m

    rho_kg = source_density * 1e3 # g/cm^3 to kg/m^3

    # Compute melting threshold fluence (energy per unit area required to heat the material from T0 to Tm)
    F_th_melt = rho_kg * cp_mix * (Tm - T0) * L_th_m/A/1e4  # J/cm^2
    
    # Compute vaporization threshold fluence (adds latent heat cost of actually vaporizing the material)
    # budget here is the energy per unit mass required to heat the material from T0 to Tv and then vaporize it
    budget = cp_mix * (Tv - T0) + Lv_mix # J/kg
    F_th_vap = rho_kg * L_th_m / A * budget / 1e4 # J/cm^2

    return F_th_melt, F_th_vap, L_th_m * 1e2 # return in J/cm^2 and effective depth in cm


def ablation_depth(F0, F_th, delta_eff_cm):
    # Compute the ablation depth based on the incident fluence F0, threshold fluence F_th, and effective depth delta_eff_cm
    if F0 < F_th:
        return 0.0
    return delta_eff_cm * np.log(F0/F_th) # cm


def calculations(m, p_partial, d, L, A_ion, n, theta_RGA, E_pulse, F_th, delta_eff_cm):
    """
    m           = molar mass of the TRACKED species (g/mol)              -- ba_m or mg_m
    p_partial   = PARTIAL density of that species in the alloy (g/cm^3)  -- ba_partial_density or mg_partial_density
    d           = spot diameter (cm)
    L           = distance from ablation target to the RGA IONIZER cage (cm)
    A_ion       = ionizer cage frontal area (cm^2)
    n           = plume forward-peaking parameter (5-15)
    theta_RGA   = angle between target surface normal and ionizer, radians
    E_pulse     = laser pulse energy (J)
    F_th        = threshold fluence to test against (J/cm^2) -- pass F_th_melt or F_th_vap
    delta_eff_cm= effective heated depth (cm), from threshold_fluence()
    """
    spot_area = np.pi * (d/2) ** 2
    F0 = E_pulse / spot_area # J/cm^2

    depth = ablation_depth(F0, F_th, delta_eff_cm)

    n_den = (p_partial/m) * n_a
    atoms_per_pulse = spot_area * n_den * depth
    emission = atoms_per_pulse * rate

    solid_angle = A_ion/L**2
    fraction_det = (n+1)/(2*np.pi)*solid_angle * np.cos(theta_RGA)**n
    flux = emission * fraction_det

    # Assuming gas velocity of 2000 m/s (2e5 cm/s) (literature uses ~ 2-3e5 cm/s)
    # Stead-State density:
    v_plume = 2e5
    n_st = flux / (v_plume * A_ion) # atoms/cm^-3

    # Pressure in torr:
    p_torr = n_st * k_b * T / 1333.322 # torr

    # Faraday-cup sensitivity of ~aa2e-4 A/Torr
    signal_current = p_torr * 2e-4 # A

    return {
        "fluence_Jcm2": F0,
        "depth_nm_per_pulse": depth * 1e7,
        "atoms_per_pulse": atoms_per_pulse,
        "flux_atoms_per_s": flux,
        "pressure_torr": p_torr,
        "signal_current_A": signal_current,
    }

In [41]:
def report(d, L, A_ion, n_plume, theta_RGA_deg, E_pulse=E_pulse_max,
           mdpp_fc=5e-11):
    theta_RGA = np.radians(theta_RGA_deg)
    F_th_melt, F_th_vap, delta_eff_cm = threshold_fluence()
    F0 = E_pulse / (np.pi * (d/2)**2)

    print(f"Threshold fluence for melting:      {F_th_melt:.3f} J/cm^2")
    print(f"Threshold fluence for vaporization: {F_th_vap:.3f} J/cm^2")
    print(f"Laser fluence at this spot size:    {F0:.3f} J/cm^2\n")

    labels = {
        "depth_nm_per_pulse": "Ablation depth (nm/pulse)",
        "atoms_per_pulse": "Ba atoms ablated per pulse",
        "flux_atoms_per_s": "Ba flux reaching ionizer (atoms/s)",
        "pressure_torr": "Predicted Ba partial pressure (Torr)",
        "signal_current_A": "Predicted Faraday cup signal (A)",
    }

    for name, F_th in (("melting threshold", F_th_melt), ("vaporization threshold", F_th_vap)):
        result = calculations(ba_m, ba_partial_density, d, L, A_ion, n_plume, theta_RGA,
                               E_pulse, F_th, delta_eff_cm)
        print(f"If real threshold = {name} ({F_th:.3f} J/cm^2):")
        for key, label in labels.items():
            print(f"  {label}: {result[key]:.3e}")
        detectable = "YES" if result["pressure_torr"] > mdpp_fc else "NO"
        print(f"  Detectable above Faraday cup floor (5e-11 Torr)? {detectable}\n")

    print(f"Biggest assumptions still unverified:")
    print(f"\n   - Absorptivity = 0.8 (no measured value for this alloy at 337.1 nm)")
    print(f"\n   - melting/vaporization temps Tm=923K, Tv=2000K (placeholders, not alloy-specific)")
    print(f"\n   - ionizer area A = 5.0 cm^2, ionizer distance L = 8.0 cm, and ionizer angle θ = 30° (inferred, not directly measured)")
    print(f"\n   - plume velocity fixed at 2e5 cm/s and forward peaking parameter n = 5 (literature-typical, not measured)")

report(d=50e-4, L=8.0, A_ion=3.0, n_plume=5, theta_RGA_deg=30.0)

Threshold fluence for melting:      0.826 J/cm^2
Threshold fluence for vaporization: 8.970 J/cm^2
Laser fluence at this spot size:    6.621 J/cm^2

If real threshold = melting threshold (0.826 J/cm^2):
  Ablation depth (nm/pulse): 2.136e+03
  Ba atoms ablated per pulse: 7.110e+12
  Ba flux reaching ionizer (atoms/s): 3.101e+12
  Predicted Ba partial pressure (Torr): 1.568e-10
  Predicted Faraday cup signal (A): 3.136e-14
  Detectable above Faraday cup floor (5e-11 Torr)? YES

If real threshold = vaporization threshold (8.970 J/cm^2):
  Ablation depth (nm/pulse): 0.000e+00
  Ba atoms ablated per pulse: 0.000e+00
  Ba flux reaching ionizer (atoms/s): 0.000e+00
  Predicted Ba partial pressure (Torr): 0.000e+00
  Predicted Faraday cup signal (A): 0.000e+00
  Detectable above Faraday cup floor (5e-11 Torr)? NO

Biggest assumptions still unverified:

   - Absorptivity = 0.8 (no measured value for this alloy at 337.1 nm)

   - melting/vaporization temps Tm=923K, Tv=2000K (placeholders, not al